# 01 · Simple IEEE 34-bus OPF Demo

对比 **Deterministic / CCOPF / Robust** 三种随机最优潮流策略。  
所有核心逻辑封装在 `src/`，本 notebook 只负责调用与结果展示。

> **网络**：34 节点链式拓扑，均匀阻抗 (r=0.01, x=0.02 pu)  
> **可再生**：PV@Bus34 + Wind@Bus20，覆盖总负荷约 70%

In [ ]:
# ── 0. 环境配置（Colab 首次运行）
# !pip install pyomo torch scikit-learn pyyaml seaborn
# !conda install -c conda-forge ipopt -y  # 或 !apt-get install -y coinor-libipopt-dev

In [ ]:
# ── 1. 路径设置 & 导入
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # 指向项目根目录

import yaml
import numpy as np
import pandas as pd

from src.dataset      import load_and_preprocess_all
from src.model        import QRLSTM
from src.train        import train_model, evaluate
from src.opf_simple   import solve_opf, P_LOAD_TOTAL, NODES
from src.monte_carlo  import run_mc, summarize
from src              import utils

# 加载配置
with open('../configs/simple_ieee34.yaml') as f:
    cfg = yaml.safe_load(f)

rc, mc_cfg, tr, opf_cfg = cfg['renewable'], cfg['monte_carlo'], cfg['training'], cfg['opf']
print("Config loaded ✓")
print(f"  P_LOAD_TOTAL = {P_LOAD_TOTAL:.2f} MW  |  N_BUSES = {len(NODES)}")

## 1 · 可再生能源参数

In [ ]:
# ── 2. 参数设定（从 yaml 读取）
pv_mu      = rc['pv_mu']
wind_mu    = rc['wind_mu']
sigma_pv   = rc['sigma_ratio'] * pv_mu
sigma_wind = rc['sigma_ratio'] * wind_mu
k          = rc['robust_sigma']
pv_low     = max(0.0, pv_mu   - k * sigma_pv)
wind_low   = max(0.0, wind_mu - k * sigma_wind)
pv_high    = pv_mu   + k * sigma_pv
wind_high  = wind_mu + k * sigma_wind

from scipy.stats import norm
z05 = norm.ppf(0.95)
slack_det = P_LOAD_TOTAL - (pv_mu + wind_mu)
slack_cc  = P_LOAD_TOTAL - max(0, pv_mu - z05*sigma_pv) - max(0, wind_mu - z05*sigma_wind)
slack_rob = P_LOAD_TOTAL - pv_low - wind_low

print(f"PV  : mu={pv_mu:.3f}  sigma={sigma_pv:.3f}  [{pv_low:.3f}, {pv_high:.3f}] MW")
print(f"Wind: mu={wind_mu:.3f}  sigma={sigma_wind:.3f}  [{wind_low:.3f}, {wind_high:.3f}] MW")
print(f"\n理论成本预验证 (100×p_slack²):")
print(f"  Det={100*slack_det**2:.1f}  CCOPF={100*slack_cc**2:.1f}  Rob={100*slack_rob**2:.1f}")
print(f"  Det < CCOPF < Rob: {100*slack_det**2 < 100*slack_cc**2 < 100*slack_rob**2}")

## 2 · QRLSTM 训练（可选，需要气象数据）

In [ ]:
# ── 3. 训练 QRLSTM（如无数据请跳过此 cell）
DATA_PATTERN = '../data/900131_*.csv'   # ← 修改为你的数据路径

import glob
if glob.glob(DATA_PATTERN):
    X_train, X_test, y_train, y_test, scaler, target_indices = \
        load_and_preprocess_all(DATA_PATTERN, window_size=tr['window_size'])

    model = QRLSTM(X_train.shape[2], hidden_size=tr['hidden_size'])
    model, loss_history = train_model(
        model, X_train, y_train,
        epochs=tr['epochs'], batch_size=tr['batch_size'], lr=tr['lr']
    )
    HAS_MODEL = True
    print("QRLSTM training complete ✓")
else:
    print("⚠ No data found — skipping QRLSTM training.")
    print("  Put CSV files in data/ and update DATA_PATTERN to enable.")
    HAS_MODEL = False

In [ ]:
# ── 4. 分位数预测可视化（需要已训练模型）
if HAS_MODEL:
    utils.plot_quantile_predictions(model, X_test, y_test, scaler, target_indices)
else:
    print("(Skipped — no model)")

## 3 · OPF 规划求解

In [ ]:
# ── 5. 三模式 OPF 求解
results = {}

print("Solving Deterministic OPF ...")
results['Deterministic'] = solve_opf(
    'det', p_pv_mu=pv_mu, p_wind_mu=wind_mu)

print("Solving CCOPF (ε=0.05) ...")
results['CCOPF'] = solve_opf(
    'ccopf',
    p_pv_mu=pv_mu,    p_wind_mu=wind_mu,
    p_pv_sigma=sigma_pv, p_wind_sigma=sigma_wind,
    epsilon=opf_cfg['epsilon'])

print("Solving Robust OPF ...")
results['Robust'] = solve_opf(
    'robust',
    p_pv_mu=pv_mu,    p_wind_mu=wind_mu,
    p_pv_low=pv_low,  p_wind_low=wind_low)

# 汇总表
rows = []
for name, res in results.items():
    rows.append({
        'Mode'        : name,
        'Cost ($)'    : f"{res['cost']:.3f}"    if res['cost']    is not None else 'N/A',
        'p_slack (MW)': f"{res['p_slack']:.4f}" if res['p_slack'] is not None else 'N/A',
        'Time (s)'    : f"{res['time_s']:.3f}",
        'Status'      : res['status'],
    })
print(pd.DataFrame(rows).set_index('Mode').to_string())

In [ ]:
# ── 6. 规划结果可视化
utils.plot_opf_cost(results)

In [ ]:
utils.plot_voltage_profile(results)

## 4 · Monte Carlo 运行验证

In [ ]:
# ── 7. Monte Carlo 验证（纯 NumPy，无 OPF 调用）
mc_stats, caps = run_mc(
    results,
    p_load_total=P_LOAD_TOTAL,
    pv_mu=pv_mu,         wind_mu=wind_mu,
    sigma_pv=sigma_pv,   sigma_wind=sigma_wind,
    n_scenarios=mc_cfg['n_scenarios'],
    n_hours=mc_cfg['n_hours'],
    k_volt=mc_cfg['k_volt'],
    cap_margin=mc_cfg['cap_margin'],
    seed=mc_cfg['seed'],
)
df_summary = summarize(mc_stats)

In [ ]:
# ── 8. MC 可视化
utils.plot_gap_rate(mc_stats)

In [ ]:
utils.plot_tradeoff(mc_stats)

In [ ]:
utils.plot_voltage_heatmap(mc_stats, method='CCOPF')

In [ ]:
utils.plot_cost_comparison(mc_stats)

In [ ]:
utils.plot_voltage_pdf(mc_stats)

## 5 · 保存结果

In [ ]:
# ── 9. 保存指标到 results/logs/metrics.json
utils.save_metrics(results, mc_stats)
print("All done ✓")